In [ ]:

# ================================================================
# SupplyChainAI — 06 LOGISTICS OPTIMIZATION
# Single-Cell Jupyter Notebook
# ================================================================
#
# FEATURES
# 01. Load delivery orders and fleet information
# 02. Generate fictional demo data when inputs are absent
# 03. Validate delivery and vehicle datasets
# 04. Geographic distance matrix (Haversine)
# 05. Capacitated vehicle routing optimization
# 06. Delivery time-window constraints
# 07. Vehicle assignment and fleet utilization
# 08. Delivery cost estimation
# 09. Fuel consumption and CO2 estimation
# 10. Route-level schedule generation
# 11. Baseline vs optimized route comparison
# 12. Delivery risk and SLA monicreate_demo_accounts.pytoring
# 13. Logistics dashboards
# 14. CSV / JSON exports
#
# Install:
# pip install pandas numpy matplotlib ortools
#
# Optional input files:
# data/delivery_orders.csv
# data/fleet.csv
#
# IMPORTANT:
# - Uses straight-line geographic distance, not road distance.
# - Travel times are estimates, not live traffic ETAs.
# - Routes are optimized for a single depot.
# - Costs and emissions are planning estimates.
# - No actual delivery or dispatch is performed.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ortools.constraint_solver import (
        pywrapcp,
        routing_enums_pb2
    )
except ImportError as exc:
    raise ImportError(
        "OR-Tools is required. Install it with: "
        "pip install ortools"
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning)

# ================================================================
# 1. CONFIGURATION
# ================================================================

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

ROOT = (
    Path("..")
    if Path("../data").exists() and not Path("data").exists()
    else Path(".")
)

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = ROOT / "reports" / "logistics_optimization"

for directory in [
    DATA_DIR,
    PROCESSED_DIR,
    REPORT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

ORDERS_PATH = DATA_DIR / "delivery_orders.csv"
FLEET_PATH = DATA_DIR / "fleet.csv"

# Fictional demo depot near Kochi, Kerala.
DEPOT = {
    "name": "Demo Kochi Distribution Center",
    "latitude": 9.9816,
    "longitude": 76.2999
}

AVERAGE_SPEED_KMPH = 30.0

# Straight-line distances are multiplied by a
# planning factor to approximate road detours.
# This is not a replacement for a routing API.
ROAD_DISTANCE_FACTOR = 1.30

DEFAULT_SERVICE_MINUTES = 10
MAX_ROUTE_MINUTES = 600
MAX_SOLVER_SECONDS = 20

FUEL_PRICE_INR_PER_LITRE = 105.0

# Illustrative diesel emissions factor.
CO2_KG_PER_LITRE = 2.68

COST_PER_KM_INR = 8.0

# All time windows are minutes after 08:00.
START_HOUR = 8

print("=" * 78)
print("SUPPLYCHAINAI — LOGISTICS OPTIMIZATION")
print("=" * 78)

# ================================================================
# 2. GENERATE SYNTHETIC DEMO DATA
# ================================================================

def generate_demo_orders(n=30):
    records = []

    for i in range(n):
        latitude = DEPOT["latitude"] + rng.uniform(
            -0.12, 0.12
        )

        longitude = DEPOT["longitude"] + rng.uniform(
            -0.12, 0.12
        )

        # Broad time windows ensure the demonstration
        # can usually find a feasible routing solution.
        window_start = int(
            rng.choice([0, 60, 120, 180])
        )

        window_end = min(
            window_start + int(
                rng.choice([240, 300, 360])
            ),
            MAX_ROUTE_MINUTES
        )

        records.append({
            "order_id": f"ORD-{i + 1:04d}",
            "customer_name": f"Demo Customer {i + 1}",
            "latitude": round(latitude, 6),
            "longitude": round(longitude, 6),
            "demand_kg": int(rng.integers(5, 45)),
            "service_minutes": DEFAULT_SERVICE_MINUTES,
            "window_start_min": window_start,
            "window_end_min": window_end,
            "priority": str(
                rng.choice(
                    ["Normal", "High", "Critical"],
                    p=[0.65, 0.25, 0.10]
                )
            )
        })

    return pd.DataFrame(records)


def generate_demo_fleet():
    return pd.DataFrame([
        {
            "vehicle_id": "VAN-001",
            "capacity_kg": 350,
            "fuel_efficiency_kmpl": 12.0,
            "fixed_cost_inr": 700
        },
        {
            "vehicle_id": "VAN-002",
            "capacity_kg": 350,
            "fuel_efficiency_kmpl": 11.5,
            "fixed_cost_inr": 700
        },
        {
            "vehicle_id": "TRUCK-001",
            "capacity_kg": 650,
            "fuel_efficiency_kmpl": 8.0,
            "fixed_cost_inr": 1200
        },
        {
            "vehicle_id": "TRUCK-002",
            "capacity_kg": 650,
            "fuel_efficiency_kmpl": 8.0,
            "fixed_cost_inr": 1200
        },
        {
            "vehicle_id": "TRUCK-003",
            "capacity_kg": 650,
            "fuel_efficiency_kmpl": 8.5,
            "fixed_cost_inr": 1200
        }
    ])


if ORDERS_PATH.exists():
    orders = pd.read_csv(ORDERS_PATH)
    orders_source = "Existing delivery orders"
else:
    orders = generate_demo_orders()
    orders.to_csv(ORDERS_PATH, index=False)
    orders_source = "Synthetic demo orders"

if FLEET_PATH.exists():
    fleet = pd.read_csv(FLEET_PATH)
    fleet_source = "Existing fleet data"
else:
    fleet = generate_demo_fleet()
    fleet.to_csv(FLEET_PATH, index=False)
    fleet_source = "Synthetic demo fleet"

print(f"\nOrders source: {orders_source}")
print(f"Fleet source: {fleet_source}")
print(f"Orders: {len(orders)}")
print(f"Vehicles: {len(fleet)}")

# ================================================================
# 3. VALIDATION
# ================================================================

ORDER_COLUMNS = [
    "order_id",
    "latitude",
    "longitude",
    "demand_kg",
    "service_minutes",
    "window_start_min",
    "window_end_min"
]

FLEET_COLUMNS = [
    "vehicle_id",
    "capacity_kg",
    "fuel_efficiency_kmpl",
    "fixed_cost_inr"
]

def require_columns(frame, columns, name):
    missing = sorted(set(columns) - set(frame.columns))

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )


require_columns(orders, ORDER_COLUMNS, "orders")
require_columns(fleet, FLEET_COLUMNS, "fleet")

for col in ORDER_COLUMNS:
    if col not in ["order_id"]:
        orders[col] = pd.to_numeric(
            orders[col], errors="coerce"
        )

for col in FLEET_COLUMNS:
    if col != "vehicle_id":
        fleet[col] = pd.to_numeric(
            fleet[col], errors="coerce"
        )

if orders[ORDER_COLUMNS].isna().any().any():
    raise ValueError(
        "Orders contain missing or invalid required values."
    )

if fleet[FLEET_COLUMNS].isna().any().any():
    raise ValueError(
        "Fleet contains missing or invalid required values."
    )

if orders["order_id"].duplicated().any():
    raise ValueError("Duplicate order IDs found.")

if fleet["vehicle_id"].duplicated().any():
    raise ValueError("Duplicate vehicle IDs found.")

if (
    (orders["demand_kg"] < 0).any()
    or (fleet["capacity_kg"] <= 0).any()
    or (fleet["fuel_efficiency_kmpl"] <= 0).any()
    or (fleet["fixed_cost_inr"] < 0).any()
):
    raise ValueError(
        "Invalid demand, capacity, fuel efficiency or cost."
    )

if (
    (orders["latitude"].abs() > 90).any()
    or (orders["longitude"].abs() > 180).any()
):
    raise ValueError("Invalid geographic coordinates.")

if (
    (orders["window_start_min"] < 0).any()
    or (
        orders["window_end_min"]
        <= orders["window_start_min"]
    ).any()
    or (
        orders["window_end_min"]
        > MAX_ROUTE_MINUTES
    ).any()
):
    raise ValueError("Invalid delivery time windows.")

if (
    orders["demand_kg"].max()
    > fleet["capacity_kg"].max()
):
    raise ValueError(
        "At least one order exceeds every vehicle capacity."
    )

if (
    orders["demand_kg"].sum()
    > fleet["capacity_kg"].sum()
):
    raise ValueError(
        "Total order weight exceeds total fleet capacity."
    )

orders = orders.reset_index(drop=True)
fleet = fleet.reset_index(drop=True)

# ================================================================
# 4. GEOGRAPHIC DISTANCES
# ================================================================

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0088

    p1 = math.radians(lat1)
    p2 = math.radians(lat2)

    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)

    a = (
        math.sin(dp / 2) ** 2
        + math.cos(p1)
        * math.cos(p2)
        * math.sin(dl / 2) ** 2
    )

    return 2 * radius * math.asin(
        min(1.0, math.sqrt(a))
    )


locations = [{
    "order_id": "DEPOT",
    "latitude": DEPOT["latitude"],
    "longitude": DEPOT["longitude"],
    "demand_kg": 0,
    "service_minutes": 0,
    "window_start_min": 0,
    "window_end_min": MAX_ROUTE_MINUTES
}]

locations.extend(
    orders.to_dict(orient="records")
)

n_locations = len(locations)

distance_matrix_km = np.zeros(
    (n_locations, n_locations),
    dtype=float
)

for i in range(n_locations):
    for j in range(n_locations):
        if i == j:
            continue

        a = locations[i]
        b = locations[j]

        distance_matrix_km[i, j] = (
            haversine_km(
                a["latitude"],
                a["longitude"],
                b["latitude"],
                b["longitude"]
            )
            * ROAD_DISTANCE_FACTOR
        )

# Integer distance units: metres.
distance_matrix_m = np.rint(
    distance_matrix_km * 1000
).astype(int)

# Integer travel time units: minutes.
travel_minutes = np.ceil(
    distance_matrix_km
    / AVERAGE_SPEED_KMPH
    * 60
).astype(int)

service_minutes = [
    int(loc["service_minutes"])
    for loc in locations
]

demand_kg = [
    int(math.ceil(loc["demand_kg"]))
    for loc in locations
]

vehicle_capacities = (
    fleet["capacity_kg"]
    .apply(math.floor)
    .astype(int)
    .tolist()
)

# ================================================================
# 5. ROUTING MODEL
# ================================================================

manager = pywrapcp.RoutingIndexManager(
    n_locations,
    len(fleet),
    0
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)

    return int(
        distance_matrix_m[from_node, to_node]
    )


distance_callback_index = (
    routing.RegisterTransitCallback(
        distance_callback
    )
)

routing.SetArcCostEvaluatorOfAllVehicles(
    distance_callback_index
)

# ================================================================
# 6. VEHICLE CAPACITY CONSTRAINTS
# ================================================================

def demand_callback(index):
    node = manager.IndexToNode(index)
    return demand_kg[node]


demand_callback_index = (
    routing.RegisterUnaryTransitCallback(
        demand_callback
    )
)

routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,
    vehicle_capacities,
    True,
    "Capacity"
)

capacity_dimension = routing.GetDimensionOrDie(
    "Capacity"
)

# ================================================================
# 7. TIME-WINDOW CONSTRAINTS
# ================================================================

def time_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)

    return int(
        travel_minutes[from_node, to_node]
        + service_minutes[from_node]
    )


time_callback_index = (
    routing.RegisterTransitCallback(
        time_callback
    )
)

routing.AddDimension(
    time_callback_index,
    MAX_ROUTE_MINUTES,
    MAX_ROUTE_MINUTES,
    False,
    "Time"
)

time_dimension = routing.GetDimensionOrDie(
    "Time"
)

for node in range(1, n_locations):
    index = manager.NodeToIndex(node)

    time_dimension.CumulVar(index).SetRange(
        int(locations[node]["window_start_min"]),
        int(locations[node]["window_end_min"])
    )

for vehicle_index in range(len(fleet)):
    start_index = routing.Start(vehicle_index)
    end_index = routing.End(vehicle_index)

    time_dimension.CumulVar(start_index).SetRange(
        0,
        0
    )

    time_dimension.CumulVar(end_index).SetRange(
        0,
        MAX_ROUTE_MINUTES
    )

    routing.AddVariableMinimizedByFinalizer(
        time_dimension.CumulVar(end_index)
    )

# ================================================================
# 8. VEHICLE FIXED COSTS
# ================================================================

# Convert INR fixed costs into an approximate
# metre-equivalent objective penalty.
#
# This balances distance and fleet usage.
# The objective is not a precise monetary cost model.

for vehicle_index, vehicle in fleet.iterrows():
    fixed_cost = int(
        vehicle["fixed_cost_inr"] * 100
    )

    routing.SetFixedCostOfVehicle(
        fixed_cost,
        int(vehicle_index)
    )

# ================================================================
# 9. SOLVER SETTINGS
# ================================================================

search_parameters = (
    pywrapcp.DefaultRoutingSearchParameters()
)

search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy
    .PARALLEL_CHEAPEST_INSERTION
)

search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic
    .GUIDED_LOCAL_SEARCH
)

search_parameters.time_limit.seconds = (
    MAX_SOLVER_SECONDS
)

search_parameters.log_search = False

print("\nSolving capacitated vehicle routing problem...")

solution = routing.SolveWithParameters(
    search_parameters
)

if solution is None:
    raise RuntimeError(
        "No feasible route found. Review fleet capacities, "
        "time windows, depot location, and route duration."
    )

print("Feasible routing solution found.")

# ================================================================
# 10. EXTRACT OPTIMIZED ROUTES
# ================================================================

def format_time(minutes):
    total = START_HOUR * 60 + int(minutes)

    hour = (total // 60) % 24
    minute = total % 60

    return f"{hour:02d}:{minute:02d}"


route_rows = []
delivery_rows = []

for vehicle_index in range(len(fleet)):
    index = routing.Start(vehicle_index)

    vehicle_id = str(
        fleet.iloc[vehicle_index]["vehicle_id"]
    )

    route_nodes = [0]
    route_distance = 0.0
    route_load = 0.0

    while not routing.IsEnd(index):
        current_node = manager.IndexToNode(index)

        next_index = solution.Value(
            routing.NextVar(index)
        )

        next_node = manager.IndexToNode(next_index)

        route_distance += (
            distance_matrix_km[current_node, next_node]
        )

        if next_node != 0:
            arrival_min = solution.Value(
                time_dimension.CumulVar(next_index)
            )

            order = locations[next_node]

            route_load += float(
                order["demand_kg"]
            )

            delivery_rows.append({
                "vehicle_id": vehicle_id,
                "order_id": order["order_id"],
                "stop_sequence": len(route_nodes),
                "arrival_minute": arrival_min,
                "estimated_arrival": format_time(
                    arrival_min
                ),
                "window_start": format_time(
                    order["window_start_min"]
                ),
                "window_end": format_time(
                    order["window_end_min"]
                ),
                "demand_kg": order["demand_kg"],
                "latitude": order["latitude"],
                "longitude": order["longitude"]
            })

        route_nodes.append(next_node)
        index = next_index

    if len(route_nodes) == 1:
        # Unused vehicle.
        continue

    end_minute = solution.Value(
        time_dimension.CumulVar(index)
    )

    vehicle = fleet.iloc[vehicle_index]

    fuel_litres = (
        route_distance
        / vehicle["fuel_efficiency_kmpl"]
    )

    fuel_cost = (
        fuel_litres
        * FUEL_PRICE_INR_PER_LITRE
    )

    distance_cost = (
        route_distance
        * COST_PER_KM_INR
    )

    total_cost = (
        vehicle["fixed_cost_inr"]
        + fuel_cost
        + distance_cost
    )

    co2_kg = (
        fuel_litres
        * CO2_KG_PER_LITRE
    )

    route_rows.append({
        "vehicle_id": vehicle_id,
        "stops": len(route_nodes) - 1,
        "route": " -> ".join(
            locations[node]["order_id"]
            for node in route_nodes
        ),
        "distance_km": round(
            route_distance, 2
        ),
        "load_kg": round(
            route_load, 2
        ),
        "capacity_kg": vehicle["capacity_kg"],
        "utilization_pct": round(
            route_load
            / vehicle["capacity_kg"]
            * 100,
            2
        ),
        "duration_minutes": end_minute,
        "fuel_litres": round(
            fuel_litres, 2
        ),
        "fuel_cost_inr": round(
            fuel_cost, 2
        ),
        "total_cost_inr": round(
            total_cost, 2
        ),
        "co2_kg": round(
            co2_kg, 2
        )
    })

routes_df = pd.DataFrame(route_rows)
deliveries_df = pd.DataFrame(delivery_rows)

if routes_df.empty:
    raise RuntimeError(
        "Solver returned no active vehicle routes."
    )

print("\nOPTIMIZED VEHICLE ROUTES")
print(routes_df.to_string(index=False))

print("\nDELIVERY SCHEDULE")
print(
    deliveries_df.head(30).to_string(index=False)
)

# ================================================================
# 11. VALIDATE SOLUTION
# ================================================================

expected_orders = set(
    orders["order_id"].astype(str)
)

assigned_orders = (
    deliveries_df["order_id"]
    .astype(str)
    .tolist()
)

if set(assigned_orders) != expected_orders:
    raise AssertionError(
        "Not all delivery orders were assigned."
    )

if len(assigned_orders) != len(set(assigned_orders)):
    raise AssertionError(
        "Duplicate delivery assignments detected."
    )

if (
    routes_df["load_kg"]
    > routes_df["capacity_kg"]
).any():
    raise AssertionError(
        "Vehicle capacity exceeded."
    )

print("\nAll orders assigned exactly once.")
print("Vehicle capacities respected.")

# ================================================================
# 12. SIMPLE BASELINE COMPARISON
# ================================================================

# Baseline:
# Every delivery receives an independent round trip
# from the depot.
#
# This is a deliberately simple reference policy,
# not an optimized or operationally validated baseline.

baseline_distance = 0.0

for node in range(1, n_locations):
    baseline_distance += (
        distance_matrix_km[0, node]
        + distance_matrix_km[node, 0]
    )

optimized_distance = float(
    routes_df["distance_km"].sum()
)

distance_savings = (
    baseline_distance - optimized_distance
)

distance_savings_pct = (
    distance_savings / baseline_distance * 100
    if baseline_distance > 0
    else 0
)

print("\nDISTANCE COMPARISON")
print(f"Independent-trip baseline: {baseline_distance:.2f} km")
print(f"Optimized routes: {optimized_distance:.2f} km")
print(f"Difference: {distance_savings:.2f} km")
print(f"Difference (%): {distance_savings_pct:.2f}%")

# ================================================================
# 13. DELIVERY SLA ANALYSIS
# ================================================================

# Feasible solver arrival times satisfy modeled
# windows, but actual road delays may differ.

deliveries_df["window_start_min"] = (
    deliveries_df["window_start"]
    .str.slice(0, 2)
    .astype(int) * 60
    +
    deliveries_df["window_start"]
    .str.slice(3, 5)
    .astype(int)
    -
    START_HOUR * 60
)

deliveries_df["window_end_min"] = (
    deliveries_df["window_end"]
    .str.slice(0, 2)
    .astype(int) * 60
    +
    deliveries_df["window_end"]
    .str.slice(3, 5)
    .astype(int)
    -
    START_HOUR * 60
)

deliveries_df["within_planned_window"] = (
    (deliveries_df["arrival_minute"]
     >= deliveries_df["window_start_min"])
    &
    (deliveries_df["arrival_minute"]
     <= deliveries_df["window_end_min"])
)

planned_sla_rate = (
    deliveries_df["within_planned_window"]
    .mean() * 100
)

# ================================================================
# 14. ROUTE MAP
# ================================================================

plt.figure(figsize=(12, 9))

plt.scatter(
    DEPOT["longitude"],
    DEPOT["latitude"],
    marker="*",
    s=260,
    label="Distribution Center"
)

colors = plt.cm.tab20(
    np.linspace(0, 1, max(len(routes_df), 1))
)

for route_index, route in routes_df.iterrows():
    vehicle_id = route["vehicle_id"]

    vehicle_stops = deliveries_df[
        deliveries_df["vehicle_id"] == vehicle_id
    ].sort_values("stop_sequence")

    longitude = [
        DEPOT["longitude"]
    ] + vehicle_stops["longitude"].tolist() + [
        DEPOT["longitude"]
    ]

    latitude = [
        DEPOT["latitude"]
    ] + vehicle_stops["latitude"].tolist() + [
        DEPOT["latitude"]
    ]

    plt.plot(
        longitude,
        latitude,
        marker="o",
        label=vehicle_id,
        color=colors[route_index]
    )

plt.title(
    "SupplyChainAI — Optimized Delivery Routes "
    "(Geographic Approximation)"
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "optimized_routes.png"
)
plt.show()

# ================================================================
# 15. ROUTE DISTANCE COMPARISON
# ================================================================

plt.figure(figsize=(11, 5))

plt.bar(
    routes_df["vehicle_id"],
    routes_df["distance_km"]
)

plt.title("Distance by Vehicle")
plt.xlabel("Vehicle")
plt.ylabel("Distance (km)")
plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "vehicle_distances.png"
)
plt.show()

# ================================================================
# 16. FLEET UTILIZATION
# ================================================================

plt.figure(figsize=(11, 5))

plt.bar(
    routes_df["vehicle_id"],
    routes_df["utilization_pct"]
)

plt.axhline(
    100,
    linestyle="--",
    label="Maximum Capacity"
)

plt.title("Fleet Capacity Utilization")
plt.xlabel("Vehicle")
plt.ylabel("Utilization (%)")
plt.legend()
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "fleet_utilization.png"
)
plt.show()

# ================================================================
# 17. TRANSPORT COST ANALYSIS
# ================================================================

plt.figure(figsize=(11, 5))

plt.bar(
    routes_df["vehicle_id"],
    routes_df["total_cost_inr"]
)

plt.title("Estimated Transportation Cost by Vehicle")
plt.xlabel("Vehicle")
plt.ylabel("Cost (INR)")
plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "transport_costs.png"
)
plt.show()

# ================================================================
# 18. EMISSIONS ANALYSIS
# ================================================================

plt.figure(figsize=(11, 5))

plt.bar(
    routes_df["vehicle_id"],
    routes_df["co2_kg"]
)

plt.title("Estimated CO2 Emissions by Vehicle")
plt.xlabel("Vehicle")
plt.ylabel("CO2 (kg)")
plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(
    REPORT_DIR / "co2_emissions.png"
)
plt.show()

# ================================================================
# 19. LOGISTICS KPIs
# ================================================================

total_cost = float(
    routes_df["total_cost_inr"].sum()
)

total_fuel = float(
    routes_df["fuel_litres"].sum()
)

total_co2 = float(
    routes_df["co2_kg"].sum()
)

vehicles_used = int(
    len(routes_df)
)

average_utilization = float(
    routes_df["utilization_pct"].mean()
)

kpis = {
    "total_orders": int(len(orders)),
    "vehicles_available": int(len(fleet)),
    "vehicles_used": vehicles_used,
    "optimized_distance_km": round(
        optimized_distance, 2
    ),
    "independent_trip_baseline_km": round(
        baseline_distance, 2
    ),
    "distance_difference_pct": round(
        distance_savings_pct, 2
    ),
    "estimated_transport_cost_inr": round(
        total_cost, 2
    ),
    "estimated_fuel_litres": round(
        total_fuel, 2
    ),
    "estimated_co2_kg": round(
        total_co2, 2
    ),
    "average_vehicle_utilization_pct": round(
        average_utilization, 2
    ),
    "planned_time_window_compliance_pct": round(
        planned_sla_rate, 2
    )
}

print("\nLOGISTICS OPTIMIZATION KPIs")
print(json.dumps(kpis, indent=2))

# ================================================================
# 20. EXPORT RESULTS
# ================================================================

routes_df.to_csv(
    PROCESSED_DIR / "optimized_routes.csv",
    index=False
)

deliveries_df.to_csv(
    PROCESSED_DIR / "optimized_delivery_schedule.csv",
    index=False
)

pd.DataFrame(
    distance_matrix_km
).to_csv(
    REPORT_DIR / "distance_matrix_km.csv",
    index=False
)

report = {
    "project": "SupplyChainAI",
    "notebook": "06_logistics_optimization.ipynb",
    "generated_at": datetime.now(
        timezone.utc
    ).isoformat(),
    "orders_source": orders_source,
    "fleet_source": fleet_source,
    "algorithm": (
        "OR-Tools Capacitated Vehicle Routing "
        "with Time Windows"
    ),
    "depot": DEPOT,
    "configuration": {
        "average_speed_kmph": AVERAGE_SPEED_KMPH,
        "road_distance_factor": ROAD_DISTANCE_FACTOR,
        "max_route_minutes": MAX_ROUTE_MINUTES,
        "fuel_price_inr_per_litre": FUEL_PRICE_INR_PER_LITRE,
        "co2_kg_per_litre": CO2_KG_PER_LITRE
    },
    "kpis": kpis,
    "limitations": [
        "Geographic distances are not actual road routes.",
        "Travel times do not include live traffic.",
        "Single-depot routing only.",
        "Vehicle capacities are modeled by weight only.",
        "Fixed vehicle costs are approximate objective penalties.",
        "Baseline uses one independent trip per order.",
        "No live GPS, dispatch or fleet tracking.",
        "No driver shift, break or regulatory constraints.",
        "Emissions and costs are estimates.",
        "All delivery schedules require operational validation."
    ]
}

with open(
    REPORT_DIR / "logistics_report.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        report,
        file,
        indent=2
    )

# ================================================================
# 21. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 78)
print("SUPPLYCHAINAI — LOGISTICS OPTIMIZATION COMPLETED")
print("=" * 78)

print(f"Delivery orders: {len(orders)}")
print(f"Vehicles used: {vehicles_used}/{len(fleet)}")
print(f"Optimized distance: {optimized_distance:.2f} km")
print(f"Estimated cost: INR {total_cost:,.2f}")
print(f"Estimated fuel: {total_fuel:.2f} litres")
print(f"Estimated CO2: {total_co2:.2f} kg")
print(f"Planned SLA compliance: {planned_sla_rate:.2f}%")

print("\nGenerated files:")
print("data/processed/optimized_routes.csv")
print("data/processed/optimized_delivery_schedule.csv")
print("reports/logistics_optimization/logistics_report.json")
print("reports/logistics_optimization/optimized_routes.png")
print("reports/logistics_optimization/vehicle_distances.png")
print("reports/logistics_optimization/fleet_utilization.png")
print("reports/logistics_optimization/transport_costs.png")
print("reports/logistics_optimization/co2_emissions.png")

print("\nNext notebook:")
print("07_inventory_optimization.ipynb")

print("=" * 78)
